# PROBABLITY DISTRIBUTION FUNCTION (PDF) FOR RNN-BASED NETWORK TRAFFIC GENERATION

In [ ]:

REQUIRED_FIELDS = ["time", "src", "dst", "protocol", "length", "info"]
VALID_PROTOCOLS = ["ZigBee", "ZigBee HA"]
def validate_packet(line):
    """
    Attempt to validate a single packet line.
    Returns tuple (is_valid, error_message, packet_dict_or_None)
    """

    original_line = line.strip()

    # Skip empty lines
    if not original_line:
        return False, "Empty line", None

    # Try JSON load
    try:
        # Remove trailing commas if present:
        cleaned = original_line.rstrip(",")
        pkt = json.loads(cleaned)
    except Exception as e:
        return False, f"JSON parse error: {e}", None

    # Check required fields
    for field in REQUIRED_FIELDS:
        if field not in pkt:
            return False, f"Missing required field: {field}", pkt

    # Detect spelling mistakes
    wrong_fields = [f for f in pkt.keys() if f not in REQUIRED_FIELDS]
    if wrong_fields:
        return False, f"Unexpected field(s): {wrong_fields}", pkt

    # # Check duplicate keys (regex)
    # duplicate_keys = re.findall(r'"(\w+)":.*"(\w+)":', original_line)
    # if duplicate_keys:
    #     return False, "Duplicate key in packet", pkt

    # Time must be a valid float
    try:
        float(pkt["time"])
    except:
        return False, "Time not numeric", pkt

    # ---- Check: length is numeric ----
    try:
        int(pkt["length"])
    except:
        return False, "Length field is not numeric", pkt

    # ---- Check: protocol must be valid ----
    if pkt["protocol"] not in VALID_PROTOCOLS:
        return False, f"Invalid protocol type: {pkt['protocol']}", pkt
    return True, None, pkt


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import seaborn as sns


## RNN PDF

In [ ]:
rnn_inter_arrival_time = []
N = list(range(1,11)) # number of trial

for n in N:
    file_path = f"Generated_Traffic/JSON_files/RNN_Exp1_Trial_{n}_generated_10_minutes.json"
    
    with open(file_path, 'r') as f:
        raw_packets = json.load(f)

    valid = []
    invalid = []

    for pkt in raw_packets:
        line = json.dumps(pkt)
        is_valid, error, fixed = validate_packet(line)
        if is_valid:
            valid.append(fixed)
        else:
            invalid.append((pkt, error))

    df = pd.DataFrame(valid)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]
    gen_inter_time = np.array(gen_inter_time)


    upper = np.percentile(gen_inter_time, 99.5)
    gen_inter_time = gen_inter_time[gen_inter_time < upper]
    rnn_inter_arrival_time.extend(gen_inter_time)


print("RNN minimum dt:", np.min(rnn_inter_arrival_time))
print("RNN maximum dt:", np.max(rnn_inter_arrival_time))
print("Zero count:", np.sum(rnn_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(rnn_inter_arrival_time) < 0)) 


## GPT 4.1 PDF

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

gpt41_inter_arrival_time = []

N = list(range(1,11)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/New_Experiments/GPT41/GPT41_Exp1_Trial_{n}_10_minute_generated_message.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    generated_traffic = pd.DataFrame(generated_traffic)


    # generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')

    gen_inter_time = generated_traffic['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gpt41_inter_arrival_time.extend(gen_inter_time)


print("GPT 4.1 minimum dt:", np.min(gpt41_inter_arrival_time))
print("GPT 4.1 maximum dt:", np.max(gpt41_inter_arrival_time))
print("Raw dt (first 20):", gpt41_inter_arrival_time[:20])
print("Zero count:", np.sum(gpt41_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gpt41_inter_arrival_time) < 0)) 


In [ ]:
# plt.figure(figsize=(8,5))

# sns.kdeplot(gpt41_inter_arrival_time, fill=True, linewidth=2, bw_adjust=5, color="green")  # bw_adjust = Adjusts Kernel (bandwidth) width
# #plt.xlim(0, rnn_dt.max()) 
# plt.xlabel("Inter-arrival time (s)", fontsize=12)
# plt.xlim(0, 40)

# plt.ylabel("PDF", fontsize=12)
# plt.title("GPT 4.1 Synthetic Traffic (EXPERIMENT 1) — Inter-arrival Time", fontsize=14)
# plt.grid(True, alpha=0.3)
# plt.savefig("GPT41_pdf_exp1.png", dpi=300, bbox_inches="tight")

# plt.show()

## GPT 5.0 PDF

In [ ]:
import pandas as pd
import numpy as np

gpt5_inter_arrival_time = []

N = list(range(1,11)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/New_Experiments/GPT5/GPT5_Exp1_Reasoning_low_Trial_{n}_10_minute_generated_message_second.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    df = pd.DataFrame(generated_traffic)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    # generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gpt5_inter_arrival_time.extend(gen_inter_time)


print("GPT 5 minimum dt:", np.min(gpt5_inter_arrival_time))
print("GPT 5 maximum dt:", np.max(gpt5_inter_arrival_time))
print("Raw dt (first 20):", gpt5_inter_arrival_time[:20])
print("Zero count:", np.sum(gpt5_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gpt5_inter_arrival_time) < 0)) 




In [ ]:
# plt.figure(figsize=(8,5))

# sns.kdeplot(gpt5_inter_arrival_time, fill=True, linewidth=2, bw_adjust=5, color="orange")  # bw_adjust = Adjusts Kernel (bandwidth) width
# plt.xlabel("Inter-arrival time (s)", fontsize=12)
# plt.xlim(0, 40)

# plt.ylabel("PDF", fontsize=12)
# plt.title("GPT 5 Synthetic Traffic (EXPERIMENT 1) — Inter-arrival Time", fontsize=14)
# plt.grid(True, alpha=0.3)
# plt.savefig("GPT5_pdf_exp1.png", dpi=300, bbox_inches="tight")

# plt.show()

## REAL TRAFFIC

In [ ]:
real_traffic = []
with open(r'Datasets/Experiment_1_one_way_communication_10_minute_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))


real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])
real_times = real_traffic["Time"]
real_dt = np.diff(real_times)

print("Real minimum dt:", np.min(real_dt))
print("Real maximum dt:", np.max(real_dt))
print("Zero count:", np.sum(real_dt == 0)) # timestamp duplication
print("Negative count:", np.sum(real_dt < 0)) 



In [ ]:
# plt.figure(figsize=(8,5))

# #plt.hist(real_dt, bins=5, density=True, alpha=0.35, color="blue", label="Histogram")

# sns.kdeplot(real_dt, fill=True, linewidth=2, bw_adjust=4, color="blue")  # bw_adjust = Adjusts Kernel (bandwidth) width
# plt.xlabel("Inter-arrival time (s)", fontsize=12)
# plt.xlim(0, 50)
# plt.ylabel("PDF", fontsize=12)
# plt.title("Real Traffic (EXPERIMENT 1) — Inter-arrival Time", fontsize=14)
# plt.grid(True, alpha=0.3)
# plt.savefig("REAL_pdf_exp1.png", dpi=300, bbox_inches="tight")

# plt.show()

## GAN

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

gan_inter_arrival_time = []

N = list(range(0,10)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/JSON_files/exp1_gan_generated_packets/generated_packets_{n}.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    df = pd.DataFrame(generated_traffic)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    # generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gan_inter_arrival_time.extend(gen_inter_time)


print("GAN minimum dt:", np.min(gan_inter_arrival_time))
print("GAN maximum dt:", np.max(gan_inter_arrival_time))
print("Zero count:", np.sum(gan_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gan_inter_arrival_time) < 0)) 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

adjust = 1
width = 4
plt.figure(figsize=(10,6))

# RNN
sns.kdeplot(rnn_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="RNN", color="red", linestyle="--")
# GAN
sns.kdeplot(gan_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GAN", color="darkmagenta", linestyle=":")
# GPT-4.1
sns.kdeplot(gpt41_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-4.1", color="green", linestyle="-", marker= "*")
# GPT-4.5
sns.kdeplot(gpt5_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-5", color="orange", linestyle="dashdot")
# Real Traffic
sns.kdeplot(real_dt, fill=False, linewidth=width, bw_adjust=adjust, label="Real Traffic", color="blue")


# X-axis only positive region
# plt.xlim(0, max(rnn_dt_clean.max(), gpt41_dt_clean.max(), real_dt.max()))

plt.xlabel("Inter-arrival Time (seconds)", fontsize=16)
plt.ylabel("PDF", fontsize=16)
# plt.title("PDF Comparison — RNN, GAN, GPT-4.1, GPT-5 and Real Traffic", fontsize=14)
plt.xlim(0, 20)
plt.ylim(0, 0.2)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("exp1_all_models_pdf.png", dpi=300, bbox_inches="tight")


plt.show()


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

adjust = 5
width = 4
plt.figure(figsize=(10,6))

# RNN
sns.kdeplot(rnn_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="RNN", color="red", linestyle="--")
# GAN
sns.kdeplot(gan_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GAN", color="darkmagenta", linestyle=":")
# GPT-4.1
sns.kdeplot(gpt41_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-4.1", color="green", linestyle="-", marker= "*")
# GPT-4.5
sns.kdeplot(gpt5_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-5", color="orange", linestyle="dashdot")
# Real Traffic
sns.kdeplot(real_dt, fill=False, linewidth=width, bw_adjust=adjust, label="Real Traffic", color="blue")

# X-axis only positive region
# plt.xlim(0, max(rnn_dt_clean.max(), gpt41_dt_clean.max(), real_dt.max()))

plt.xlabel("Inter-arrival Time (seconds)", fontsize=16)
plt.ylabel("PDF", fontsize=16)
# plt.title("PDF Comparison — RNN, GAN, GPT-4.1, GPT-5 and Real Traffic", fontsize=14)
plt.xlim(0, 20)
plt.ylim(0, 0.10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("exp1_all_models_pdf_close.png", dpi=300, bbox_inches="tight")

plt.show()


# EXPERIMENT 2

## REAL TRAFFIC

In [ ]:
real_traffic = []

with open(r'Datasets/Experiment_2_input_sample.json', 'r') as file:
    for line in file:
        real_traffic.append(json.loads(line))


real_traffic  = pd.DataFrame(real_traffic)
real_traffic = real_traffic.drop(columns=["No.", "Info_clean"])
real_times = real_traffic["Time"]
real_dt = np.diff(real_times)

print("Real minimum dt:", np.min(real_dt))
print("Real maximum dt:", np.max(real_dt))
print("Zero count:", np.sum(real_dt == 0)) # timestamp duplication
print("Negative count:", np.sum(real_dt < 0)) 



## RNN

In [ ]:
rnn_inter_arrival_time = []
N = list(range(1,11)) # number of trial

for n in N:
    file_path = f"Generated_Traffic/JSON_files/RNN_Exp2_Trial_{n}_generated_10_minutes.json"
    
    with open(file_path, 'r') as f:
        raw_packets = json.load(f)

    valid = []
    invalid = []

    for pkt in raw_packets:
        line = json.dumps(pkt)
        is_valid, error, fixed = validate_packet(line)
        if is_valid:
            valid.append(fixed)
        else:
            invalid.append((pkt, error))

    df = pd.DataFrame(valid)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]
    gen_inter_time = np.array(gen_inter_time)


    upper = np.percentile(gen_inter_time, 99.5)
    gen_inter_time = gen_inter_time[gen_inter_time < upper]
    rnn_inter_arrival_time.extend(gen_inter_time)


print("RNN minimum dt:", np.min(rnn_inter_arrival_time))
print("RNN maximum dt:", np.max(rnn_inter_arrival_time))
print("Zero count:", np.sum(rnn_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(rnn_inter_arrival_time) < 0)) 


## GPT41

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

gpt41_inter_arrival_time = []

N = list(range(1,11)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/New_Experiments/GPT41/GPT41_Exp2_Trial_{n}_10_minute_generated_message.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    df = pd.DataFrame(generated_traffic)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)



    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gpt41_inter_arrival_time.extend(gen_inter_time)


print("GPT 4.1 minimum dt:", np.min(gpt41_inter_arrival_time))
print("GPT 4.1 maximum dt:", np.max(gpt41_inter_arrival_time))
print("Zero count:", np.sum(gpt41_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gpt41_inter_arrival_time) < 0)) 


## GPT 5 

In [ ]:
import pandas as pd
import numpy as np

gpt5_inter_arrival_time = []

N = list(range(1,11)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/New_Experiments/GPT5/GPT5_Exp2_Reasoning_low_Trial_10_10_minute_generated_message.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    df = pd.DataFrame(generated_traffic)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    # generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gpt5_inter_arrival_time.extend(gen_inter_time)


print("GPT 5 minimum dt:", np.min(gpt5_inter_arrival_time))
print("GPT 5 maximum dt:", np.max(gpt5_inter_arrival_time))
print("Raw dt (first 20):", gpt5_inter_arrival_time[:20])
print("Zero count:", np.sum(gpt5_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gpt5_inter_arrival_time) < 0)) 




## GAN

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import jensenshannon

gan_inter_arrival_time = []

N = list(range(0,10)) # number of trial
for n in N:

    file_path = fr"Generated_Traffic/JSON_files/exp2_gan_generated_packets/generated_packets_{n}.json"

    with open(file_path, 'r') as f:
        generated_traffic = json.load(f)
    df = pd.DataFrame(generated_traffic)

    # --- CLEAN & SORT TIME ---
    df['time'] = pd.to_numeric(df['time'], errors='coerce')
    df = df.dropna(subset=['time'])  # remove NaN time rows
    df = df.sort_values(by='time').reset_index(drop=True)

    # generated_traffic['time'] = pd.to_numeric( generated_traffic['time'], errors='coerce')

    gen_inter_time = df['time'].astype(float).diff()
    gen_inter_time = [x for x in gen_inter_time if not (isinstance(x, float) and np.isnan(x))]

    gan_inter_arrival_time.extend(gen_inter_time)


print("GAN minimum dt:", np.min(gan_inter_arrival_time))
print("GAN maximum dt:", np.max(gan_inter_arrival_time))
print("Zero count:", np.sum(gan_inter_arrival_time == 0)) # timestamp duplication
print("Negative count:", np.sum( np.array(gan_inter_arrival_time) < 0)) 


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

adjust = 1
width = 4
plt.figure(figsize=(10,6))

# RNN
sns.kdeplot(rnn_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="RNN", color="red", linestyle="--")
# GAN
sns.kdeplot(gan_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GAN", color="darkmagenta", linestyle=":")
# GPT-4.1
sns.kdeplot(gpt41_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-4.1", color="green", linestyle="-", marker= "*")
# GPT-4.5
sns.kdeplot(gpt5_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-5", color="orange", linestyle="dashdot")
# Real Traffic
sns.kdeplot(real_dt, fill=False, linewidth=width, bw_adjust=adjust, label="Real Traffic", color="blue")


plt.xlabel("Inter-arrival Time (seconds)", fontsize=16)
plt.ylabel("PDF", fontsize=16)
# plt.title("PDF Comparison — RNN, GAN, GPT-4.1, GPT-5 and Real Traffic", fontsize=14)
plt.xlim(0, 20)
plt.ylim(0, 0.2)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("exp2_all_models_pdf.png", dpi=300, bbox_inches="tight")
plt.show()



In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

adjust = 5
width = 4
plt.figure(figsize=(10,6))

# RNN
sns.kdeplot(rnn_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="RNN", color="red", linestyle="--")
# GAN
sns.kdeplot(gan_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GAN", color="darkmagenta", linestyle=":")
# GPT-4.1
sns.kdeplot(gpt41_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-4.1", color="green", linestyle="-", marker= "*")
# GPT-4.5
sns.kdeplot(gpt5_inter_arrival_time, fill=False, linewidth=width, bw_adjust=adjust, label="GPT-5", color="orange", linestyle="dashdot")
# Real Traffic
sns.kdeplot(real_dt, fill=False, linewidth=width, bw_adjust=adjust, label="Real Traffic", color="blue")

# X-axis only positive region
# plt.xlim(0, max(rnn_dt_clean.max(), gpt41_dt_clean.max(), real_dt.max()))

plt.xlabel("Inter-arrival Time (seconds)", fontsize=16)
plt.ylabel("PDF", fontsize=16)
# plt.title("PDF Comparison — RNN, GAN, GPT-4.1, GPT-5 and Real Traffic", fontsize=14)
plt.xlim(0, 20)
plt.ylim(0, 0.10)
plt.xticks(fontsize=12)
plt.yticks(fontsize=12)
plt.legend(fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()

plt.savefig("exp2_all_models_pdf_close.png", dpi=300, bbox_inches="tight")

plt.show()
